In [6]:
import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint

from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss

ENCODER_LEN = 168
DECODER_LEN = 120
VAL_START = "2023-01-01"   # last year of train period held out for early stopping

WEATHER_VARS = [
    "temperature_2m", "relative_humidity_2m", "dew_point_2m",
    "apparent_temperature", "precipitation", "snowfall",
    "cloud_cover", "surface_pressure", "wind_speed_10m", "wind_gusts_10m",
    "shortwave_radiation", "direct_radiation", "diffuse_radiation",
]
UNKNOWN_REALS = ["demand", "net_generation", "total_interchange"] + \
                [f"act_{v}" for v in WEATHER_VARS]
KNOWN_REALS = [f"fc_{v}" for v in WEATHER_VARS] + \
              ["fx_app_roll72", "fx_cdh_24h", "fx_hot_streak_day",
               "fx_night_min_app_prev", "time_idx"]
KNOWN_CATS = ["hour", "day_of_week", "day_of_month", "month", "is_holiday"]
# NOTE: fc_is_synthetic deliberately EXCLUDED (constant 1 in train, 0 in test)


def load(csv):
    df = pd.read_csv(csv, parse_dates=["ts"])
    for c in KNOWN_CATS:
        df[c] = df[c].astype(str).astype("category")
    df["series"] = df["series"].astype(str)
    return df.sort_values("time_idx").reset_index(drop=True)


def build_datasets(df):
    val_start_idx = int(df.loc[df["ts"] >= VAL_START, "time_idx"].min())
    test_start_idx = int(df.loc[df["split"] == "test", "time_idx"].min())

    train_df = df[df["time_idx"] < val_start_idx]

    training = TimeSeriesDataSet(
        train_df,
        time_idx="time_idx",
        target="demand",
        group_ids=["series"],
        max_encoder_length=ENCODER_LEN,
        max_prediction_length=DECODER_LEN,
        time_varying_unknown_reals=UNKNOWN_REALS,
        time_varying_known_reals=KNOWN_REALS,
        time_varying_known_categoricals=KNOWN_CATS,
        target_normalizer=GroupNormalizer(groups=["series"]),
        add_relative_time_idx=True,
        add_target_scales=True,
        allow_missing_timesteps=False,
    )

    # validation: origins strictly after train cutoff, before test
    val_df = df[df["time_idx"] < test_start_idx]
    validation = TimeSeriesDataSet.from_dataset(
        training, val_df, min_prediction_idx=val_start_idx,
        stop_randomization=True)

    # test: origins strictly inside the genuine-forecast test window
    test = TimeSeriesDataSet.from_dataset(
        training, df, min_prediction_idx=test_start_idx,
        stop_randomization=True)

    return training, validation, test, test_start_idx


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--csv", default="data/processed/tft_clean.csv")
    ap.add_argument("--max-epochs", type=int, default=40)
    ap.add_argument("--batch", type=int, default=64)
    args, _ = ap.parse_known_args()

    pl.seed_everything(42)
    df = load(args.csv)
    training, validation, test, test_start_idx = build_datasets(df)

    train_dl = training.to_dataloader(train=True, batch_size=args.batch,
                                      num_workers=4)
    val_dl = validation.to_dataloader(train=False, batch_size=args.batch * 4,
                                      num_workers=4)
    test_dl = test.to_dataloader(train=False, batch_size=args.batch * 4,
                                 num_workers=4)

    model = TemporalFusionTransformer.from_dataset(
        training,
        hidden_size=160,
        attention_head_size=4,
        dropout=0.1,
        hidden_continuous_size=80,
        loss=QuantileLoss(quantiles=[0.1, 0.5, 0.9]),
        learning_rate=1e-3,
        log_interval=200,
    )
    print(f"Model parameters: {model.size() / 1e6:.1f}M")

    ckpt = ModelCheckpoint(dirpath="checkpoints", filename="tft_clean_best",
                           monitor="val_loss", mode="min")
    trainer = pl.Trainer(
        max_epochs=args.max_epochs,
        accelerator="auto",
        gradient_clip_val=0.01,
        callbacks=[EarlyStopping(monitor="val_loss", patience=5), ckpt],
        enable_progress_bar=True,
    )
    trainer.fit(model, train_dl, val_dl)

    # ---------------- evaluation: Day 1-5 MAPE on the test window ----------
    best = TemporalFusionTransformer.load_from_checkpoint(ckpt.best_model_path)
    preds = best.predict(test_dl, mode="prediction", return_y=True,
                         trainer_kwargs={"accelerator": "auto"})
    y_hat = preds.output.cpu().numpy()          # (n_samples, 120) P50
    y_true = preds.y[0].cpu().numpy()           # (n_samples, 120)

    metrics = {}
    for day in range(1, 6):
        s, e = (day - 1) * 24, day * 24
        mape = float(np.mean(np.abs(y_true[:, s:e] - y_hat[:, s:e])
                             / np.clip(np.abs(y_true[:, s:e]), 1e-6, None)) * 100)
        metrics[f"day{day}_mape"] = round(mape, 3)
        print(f"Day {day} test MAPE: {mape:.2f}%   (baseline: "
              f"{[3.84, 4.07, 4.12, 4.14, 4.16][day - 1]:.2f}%)")
    metrics["overall_mape"] = round(float(
        np.mean(np.abs(y_true - y_hat) / np.clip(np.abs(y_true), 1e-6, None)) * 100), 3)
    metrics["best_checkpoint"] = ckpt.best_model_path
    metrics["n_test_samples"] = int(y_true.shape[0])

    Path("metrics_tft_clean.json").write_text(json.dumps(metrics, indent=2))
    print(json.dumps(metrics, indent=2))


if __name__ == "__main__":
    main()

Epoch 5/39 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1023/1023 0:08:55 • 0:00:00 1.91it/s v_num: 0.000 train_loss_step:     
                                                                                 75.692 val_loss: 351.994          
                                                                                 train_loss_epoch: 79.049          

/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'loss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['loss'])`.
/opt/app-root/lib64/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:213: Attribute 'logging_metrics' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['logging_metrics'])`.


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Day 1 test MAPE: 4.36%   (baseline: 3.84%)
Day 2 test MAPE: 4.49%   (baseline: 4.07%)
Day 3 test MAPE: 4.51%   (baseline: 4.12%)
Day 4 test MAPE: 4.52%   (baseline: 4.14%)
Day 5 test MAPE: 4.52%   (baseline: 4.16%)
{
  "day1_mape": 4.355,
  "day2_mape": 4.486,
  "day3_mape": 4.507,
  "day4_mape": 4.516,
  "day5_mape": 4.523,
  "overall_mape": 4.478,
  "best_checkpoint": "/opt/app-root/src/Forecasting-Energy-Demand/Sangar/checkpoints/tft_clean_best.ckpt",
  "n_test_samples": 20465
}


In [8]:
import os
for p in [CSV_CLEAN, CSV_LEADS, CKPT]:
    print(os.path.exists(p), p)

NameError: name 'CSV_CLEAN' is not defined